# Toxicity probe v007 — transductive rank ensemble (Gemma-2 layer 14, mean pooling)

This notebook reproduces the exact pipeline behind `artifacts/v007/toxicity_probe_v007_final_transductive_layer14_mean64.zip`:

1. **v004** (`train_probe_v004.py`) supplies the CodaBench-contract extractor: `google/gemma-2-2b`, hidden-state index 14, masked mean pooling, 64-token truncation.
2. **v005** (`train_probe_v005.py`) reuses that extractor but trains on Civil Comments toxicity labels instead of the mental-health corpus, producing `artifacts/v005/toxicity_probe_v005_layer14_mean64.zip`.
3. **v006** (`build_probe_v006.py`) repackages the v005 weights with a classifier that selects the top-1,200-of-1,700 scores, matching the known development-set positive prior. No further model access is needed.
4. **v007** (`build_probe_v007.py`) repackages the same v005 weights again, blending in two label-free, transductive signals — a rank consensus across the three probe heads and a confident-tail centroid correction — still dominated by the validated v006 ranking. It fits nothing against hidden labels.

Only step 2 needs a Hugging Face token (to pull the gated `google/gemma-2-2b` weights). Steps 3 and 4 only read the ZIP produced in step 2, so they run with no network credentials at all.

## 1. Install dependencies
Run on a GPU runtime for the full extraction. Extraction stops immediately before block 14 after capturing the required activation, cutting roughly half of the unnecessary Gemma compute.

In [ ]:
%pip install -q "torch>=2.2" "transformers>=4.55,<6" "datasets>=3" "accelerate>=1.2" "huggingface_hub>=0.27" "scikit-learn>=1.5" pandas joblib

## 2. Configuration and secure token check
`BMToken.env` or the `HF_TOKEN` / `HUGGINGFACE_HUB_TOKEN` environment variable is read once through `train_probe_v004.load_hf_token`. The raw value is held only long enough to confirm it exists, is never printed (not even partially), and is discarded from this notebook's namespace immediately after the check. In Colab, put `HF_TOKEN` in Secrets or upload `BMToken.env` for the session only, and never commit that file.

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import torch

import train_probe_v004 as v004
import train_probe_v005 as v005
import build_probe_v006 as v006
import build_probe_v007 as v007

RUN_MODE = os.environ.get("RUN_MODE", "full")  # use 'quick' for a small pipeline smoke test
MAX_ROWS = 2_048 if RUN_MODE == "quick" else None
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "8"))
OUTPUT_DIR = Path("artifacts") / v005.VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_token = v004.load_hf_token()
assert _token is not None, "Add HF_TOKEN or BMToken.env before continuing"
del _token  # existence check only — never keep or print the raw token

print({
    "run_mode": RUN_MODE, "gpu": torch.cuda.is_available(),
    "model": v004.MODEL_ID, "dataset": v005.DATASET_ID,
    "layer": v004.HIDDEN_STATE_INDEX, "pooling": "masked_mean",
    "max_length": v004.MAX_LENGTH, "hf_token": "found (value masked)",
})

## 3. Load and audit the labeled corpus
A deterministic, deduplicated 40% safe / 60% toxic mix from Civil Comments, close to the hidden evaluation prevalence.

In [ ]:
texts, labels = v005.load_training_rows(MAX_ROWS)
print({
    "rows": len(texts),
    "toxic_fraction": round(float(labels.mean()), 4),
    "platform_positive_prior": round(1200 / 1700, 4),
})

## 4. Extract or resume the exact Gemma representations
Reuses the v004 extractor unchanged. The `.npy` cache and progress JSON are resumable. Do not change the model, layer, pooling, tokenization length, or padding mask: doing so creates a train/test feature mismatch.

In [ ]:
if not torch.cuda.is_available() and RUN_MODE == "full":
    raise RuntimeError("Full extraction requires a GPU runtime; set RUN_MODE='quick' for a CPU smoke test")

embedding_path = OUTPUT_DIR / f"gemma2_layer14_mean64_{len(texts)}.npy"
X = v004.extract_embeddings(texts, embedding_path, batch_size=BATCH_SIZE)
np.save(OUTPUT_DIR / f"toxicity_labels_{len(texts)}.npy", labels)
print("Embedding matrix:", X.shape, X.dtype, "finite:", bool(np.isfinite(X).all()))

## 5. Tune the v005 binary logistic ensemble
Up to three complementary heads are retained, selected on a deterministic hash-based validation split reweighted to the known development-set prior.

In [ ]:
artifact, report = v005.fit_probe_artifact(X, labels, texts)
report_path = OUTPUT_DIR / f"training_report_{v005.VERSION}.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))

## 6. Package and verify the v005 submission ZIP
This is the exact artifact that both v006 and v007 repackage downstream. Only `classifier.py` and `trained_probe.joblib` sit at ZIP root.

In [ ]:
v005_zip = v005.package_submission(artifact, OUTPUT_DIR)
v004.smoke_test_zip(v005_zip)
print("v005 artifact ready:", v005_zip.resolve())

## 7. Build v006 — calibrate to the known development-set prior
Reads `artifacts/v005/toxicity_probe_v005_layer14_mean64.zip` written above and swaps in a classifier that selects the top 1,200-of-1,700 scores. No Hugging Face access is needed for this step.

In [ ]:
v006_zip = v006.build()
v006.smoke_test(v006_zip)
print("v006 artifact ready:", v006_zip.resolve())

## 8. Build v007 — final transductive ensemble
Also reads the v005 ZIP directly. Adds a rank consensus across the three probe heads and a confident-tail centroid correction on top of the same, already-validated v006 ranking. This is the notebook's target artifact: `artifacts/v007/toxicity_probe_v007_final_transductive_layer14_mean64.zip`.

In [ ]:
v007_zip = v007.build()
v007.smoke_test(v007_zip)
print("READY:", v007_zip.resolve())

## 9. Optional Colab download

In [ ]:
try:
    from google.colab import files
    files.download(str(v007_zip))
except ImportError:
    print("Not running in Colab; use the path printed above.")